In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
!pip install -q \
    "torch>=2.0.0" \
    "transformers>=4.40.0" \
    "peft>=0.10.0" \
    "datasets>=2.18.0" \
    "accelerate>=0.28.0" \
    "sentencepiece>=0.2.0" \
    "protobuf>=3.20.0" \
    "scipy>=1.11.0" \
    "scikit-learn>=1.3.0" \
    "tqdm>=4.66.0"

In [25]:
"""
generate_dataset.py  –  Kaggle Version
---------------------------------------
Synthetically generates 600 (NL command, JSON API call) pairs and saves them
to /kaggle/working/dataset.jsonl in the T5 input/target format.
"""

import json
import random

random.seed(42)

# ── Building blocks ───────────────────────────────────────────────────────────
NAMES = [
    "Alice", "Bob", "Carol", "David", "Eve", "Frank", "Grace", "Henry",
    "Iris", "Jack", "Karen", "Leo", "Maria", "Nathan", "Olivia", "Paul",
    "Quinn", "Rachel", "Steve", "Tina", "Uma", "Victor", "Wendy", "Xavier",
]

MEETING_TITLES = [
    "standup", "sync", "review", "planning session", "retrospective",
    "kickoff", "demo", "interview", "1:1", "brainstorm", "strategy session",
    "all-hands", "workshop", "presentation", "check-in",
]

REMINDER_TASKS = [
    "call {name}", "email {name}", "follow up with {name}", "text {name}",
    "send report", "submit invoice", "review PR", "pay bills",
    "buy groceries", "pick up dry cleaning", "water the plants",
    "take medication", "exercise", "read emails", "prepare slides",
]

DAYS_OF_WEEK  = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
RELATIVE_DAYS = ["today", "tomorrow", "next Monday", "next Tuesday",
                 "next Wednesday", "next Thursday", "next Friday"]

HOURS_12 = [
    ("9am",  "09:00"), ("10am", "10:00"), ("11am", "11:00"),
    ("noon", "12:00"), ("1pm",  "13:00"), ("2pm",  "14:00"),
    ("3pm",  "15:00"), ("4pm",  "16:00"), ("5pm",  "17:00"),
    ("6pm",  "18:00"), ("7pm",  "19:00"), ("8pm",  "20:00"),
]

DURATIONS = [
    ("30 minutes", 30), ("an hour", 60), ("45 minutes", 45),
    ("90 minutes", 90), ("2 hours", 120),
]


def random_name(exclude=None):
    choices = [n for n in NAMES if n != exclude]
    return random.choice(choices)

def random_time():
    return random.choice(HOURS_12)

def random_day():
    return random.choice(RELATIVE_DAYS)

def random_attendees(n=None):
    count = n or random.randint(2, 5)
    return random.sample(NAMES, count)


# ── Template generators ───────────────────────────────────────────────────────

def gen_insert_meeting():
    attendees = random_attendees(random.randint(1, 3))
    time_str, time_24 = random_time()
    day = random_day()
    title = random.choice(MEETING_TITLES)
    duration_str, duration_min = random.choice(DURATIONS)

    nl_templates = [
        f"Schedule a {title} with {', '.join(attendees)} {day} at {time_str}",
        f"Book a {title} with {attendees[0]} for {duration_str} {day} at {time_str}",
        f"Set up a {title} with {', '.join(attendees)} at {time_str} {day}",
        f"Arrange a {duration_str} {title} with {attendees[0]} {day} at {time_str}",
        f"Create a calendar event for a {title} with {', '.join(attendees)} {day} at {time_str}",
        f"Add a {title} to my calendar {day} at {time_str} with {attendees[0]}",
        f"I need to meet with {attendees[0]} for a {title} {day} at {time_str}",
        f"Put a {title} on my schedule for {day} at {time_str}",
    ]

    nl = random.choice(nl_templates)
    target = {
        "action": "insert",
        "title": title.title(),
        "attendees": attendees,
        "start_time": time_24,
        "duration_minutes": duration_min,
        "day": day,
    }
    return nl, target


def gen_delete_event():
    title = random.choice(MEETING_TITLES)
    time_str, time_24 = random_time()
    day = random_day()

    nl_templates = [
        f"Cancel my {title} {day} at {time_str}",
        f"Delete the {title} scheduled for {day} at {time_str}",
        f"Remove the {title} from my calendar on {day}",
        f"Cancel the {time_str} {title} on {day}",
        f"I need to cancel my {title} {day}",
        f"Clear my {title} slot at {time_str} on {day}",
        f"Delete my {title} appointment {day}",
    ]

    nl = random.choice(nl_templates)
    target = {
        "action": "delete",
        "title": title.title(),
        "start_time": time_24,
        "day": day,
    }
    return nl, target


def gen_update_event():
    title = random.choice(MEETING_TITLES)
    old_time_str, old_time_24 = random_time()
    new_time_str, new_time_24 = random_time()
    while new_time_24 == old_time_24:
        new_time_str, new_time_24 = random_time()
    day = random_day()

    nl_templates = [
        f"Move my {title} from {old_time_str} to {new_time_str} on {day}",
        f"Reschedule the {title} at {old_time_str} to {new_time_str} on {day}",
        f"Change my {title} on {day} from {old_time_str} to {new_time_str}",
        f"Update the {title} scheduled at {old_time_str} {day} to {new_time_str}",
        f"Push my {title} on {day} from {old_time_str} to {new_time_str}",
        f"Shift the {old_time_str} {title} on {day} to {new_time_str}",
    ]

    nl = random.choice(nl_templates)
    target = {
        "action": "update",
        "title": title.title(),
        "old_start_time": old_time_24,
        "new_start_time": new_time_24,
        "day": day,
    }
    return nl, target


def gen_set_reminder():
    name = random_name()
    task_template = random.choice(REMINDER_TASKS)
    task = task_template.format(name=name)
    time_str, time_24 = random_time()
    day = random_day()

    nl_templates = [
        f"Remind me to {task} at {time_str} {day}",
        f"Set a reminder to {task} {day} at {time_str}",
        f"Alert me to {task} at {time_str} on {day}",
        f"Don't let me forget to {task} at {time_str} {day}",
        f"Add a reminder for {task} at {time_str} {day}",
        f"Ping me to {task} {day} at {time_str}",
    ]

    nl = random.choice(nl_templates)
    target = {
        "action": "reminder",
        "task": task,
        "time": time_24,
        "day": day,
    }
    return nl, target


def gen_query_schedule():
    day = random_day()
    time_str, time_24 = random_time()

    nl_templates = [
        f"What's on my calendar {day}?",
        f"Do I have anything scheduled for {day}?",
        f"What meetings do I have {day}?",
        f"Am I free at {time_str} {day}?",
        f"Show me my schedule for {day}",
        f"What events do I have {day}?",
        f"Check my availability {day} at {time_str}",
        f"Is {day} at {time_str} free?",
    ]

    nl = random.choice(nl_templates)
    target = {
        "action": "query",
        "day": day,
        "time": time_24 if "free" in nl or "availability" in nl else None,
    }
    target = {k: v for k, v in target.items() if v is not None}
    return nl, target


def gen_add_attendee():
    title = random.choice(MEETING_TITLES)
    name = random_name()
    day = random_day()
    time_str, time_24 = random_time()

    nl_templates = [
        f"Add {name} to my {title} {day} at {time_str}",
        f"Invite {name} to the {title} on {day}",
        f"Include {name} in the {time_str} {title} {day}",
        f"Add {name} as an attendee to the {title} on {day} at {time_str}",
        f"Can you invite {name} to my {title} {day}?",
    ]

    nl = random.choice(nl_templates)
    target = {
        "action": "add_attendee",
        "title": title.title(),
        "attendee": name,
        "start_time": time_24,
        "day": day,
    }
    return nl, target


# ── Registry ──────────────────────────────────────────────────────────────────
GENERATORS = [
    (gen_insert_meeting, 0.35),
    (gen_delete_event,   0.15),
    (gen_update_event,   0.15),
    (gen_set_reminder,   0.20),
    (gen_query_schedule, 0.10),
    (gen_add_attendee,   0.05),
]
FUNCS, WEIGHTS = zip(*GENERATORS)


def generate_dataset(n: int = 600) -> list:
    examples = []
    prefix = "translate English to Calendar API: "
    seen_inputs = set()
    attempts = 0

    while len(examples) < n and attempts < n * 10:
        attempts += 1
        gen_fn = random.choices(FUNCS, weights=WEIGHTS, k=1)[0]
        nl, target = gen_fn()

        input_text = prefix + nl
        if input_text in seen_inputs:
            continue
        seen_inputs.add(input_text)

        target_text = json.dumps(target, separators=(",", ":"))
        examples.append({"input_text": input_text, "target_text": target_text})

    return examples


# ── KAGGLE: output path hardcoded to /kaggle/working/ ────────────────────────
def main():
    print("Generating dataset…")
    examples = generate_dataset(600)

    out_path = "/kaggle/working/dataset.jsonl"          # <-- Kaggle path

    with open(out_path, "w", encoding="utf-8") as f:
        for ex in examples:
            f.write(json.dumps(ex) + "\n")

    print(f"✅  Saved {len(examples)} examples to {out_path}")

    print("\n── Sample examples ──────────────────────────────────────────")
    for ex in random.sample(examples, 3):
        print("INPUT :", ex["input_text"])
        print("TARGET:", ex["target_text"])
        print()


main()   # called directly so it runs when pasted into a Kaggle cell

Generating dataset…
✅  Saved 600 examples to /kaggle/working/dataset.jsonl

── Sample examples ──────────────────────────────────────────
INPUT : translate English to Calendar API: Do I have anything scheduled for next Thursday?
TARGET: {"action":"query","day":"next Thursday"}

INPUT : translate English to Calendar API: Don't let me forget to follow up with Steve at 5pm next Monday
TARGET: {"action":"reminder","task":"follow up with Steve","time":"17:00","day":"next Monday"}

INPUT : translate English to Calendar API: Put a 1:1 on my schedule for next Monday at 2pm
TARGET: {"action":"insert","title":"1:1","attendees":["Bob"],"start_time":"14:00","duration_minutes":90,"day":"next Monday"}



In [26]:
"""
train.py  –  Kaggle Version (Complex Data Support)
-----------------------------
Updated to handle longer attendee lists and complex JSON patterns.
"""

import os
import math

import torch
from datasets import load_dataset, DatasetDict
from transformers import (
    T5ForConditionalGeneration,
    T5TokenizerFast,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from peft import LoraConfig, TaskType, get_peft_model

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_NAME    = "t5-small"
DATASET_PATH  = "/kaggle/working/dataset.jsonl"
OUTPUT_DIR    = "/kaggle/working/lora-adapter"

# 1. INCREASED LENGTHS: Complex JSON and long attendee lists take up more tokens.
# 256 is safer for lists of 5+ people.
MAX_INPUT_LEN  = 256 
MAX_TARGET_LEN = 256 

BATCH_SIZE    = 8
GRAD_ACCUM    = 4
NUM_EPOCHS    = 50
LEARNING_RATE = 3e-4
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1
VAL_SPLIT     = 0.1

# LoRA hyper-parameters
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
LORA_TARGET  = ["q", "v"]


# ── Helpers ───────────────────────────────────────────────────────────────────

def load_and_split(path: str, val_fraction: float = VAL_SPLIT):
    raw   = load_dataset("json", data_files={"train": path})["train"]
    split = raw.train_test_split(test_size=val_fraction, seed=42)
    return DatasetDict({"train": split["train"], "validation": split["test"]})


def tokenize_fn(examples, tokenizer):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=MAX_INPUT_LEN,
        padding="max_length",
        truncation=True,
    )
    
    # Use text_target for compatibility with modern Transformers [cite: 25]
    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=MAX_TARGET_LEN,
        padding="max_length",
        truncation=True,
    )

    label_ids = [
        [(t if t != tokenizer.pad_token_id else -100) for t in lab]
        for lab in labels["input_ids"]
    ]
    model_inputs["labels"] = label_ids
    return model_inputs


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    # 1. Dataset
    print("Loading dataset…")
    ds = load_and_split(DATASET_PATH)
    print(f"  Train: {len(ds['train'])}  |  Val: {len(ds['validation'])}")

    # 2. Tokeniser
    print("Loading tokenizer…")
    tokenizer = T5TokenizerFast.from_pretrained(MODEL_NAME)

    print("Tokenising dataset…")
    tokenised = ds.map(
        lambda ex: tokenize_fn(ex, tokenizer),
        batched=True,
        remove_columns=ds["train"].column_names,
    )
    tokenised.set_format(type="torch")

    # 3. Base model
    print("Loading base model…")
    base_model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

    # 4. Wrap with LoRA
    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET,
        lora_dropout=LORA_DROPOUT,
        bias="none",
    )
    model = get_peft_model(base_model, lora_cfg)
    model.print_trainable_parameters()

    # 5. Data collator
    collator = DataCollatorForSeq2Seq(
        tokenizer,
        model=model,
        label_pad_token_id=-100,
        pad_to_multiple_of=8,
    )

    # 6. Training arguments
    training_args = Seq2SeqTrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LEN,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_dir=os.path.join(OUTPUT_DIR, "logs"),
        logging_steps=10, # More frequent logging to see progress better
        fp16=torch.cuda.is_available(),
        report_to="none",
        save_total_limit=2,
        hub_model_id=None,
    )

    # 7. Trainer
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenised["train"],
        eval_dataset=tokenised["validation"],
        processing_class=tokenizer,
        data_collator=collator,
        # 2. INCREASED PATIENCE: Complex tasks take longer to "click." 
        # Don't stop training too early.
        callbacks=[EarlyStoppingCallback(early_stopping_patience=10)], 
    )

    # 8. Train
    print("\n🚀  Starting training…")
    trainer.train()

    # 9. Save LoRA adapter only
    print(f"\n💾  Saving LoRA adapter to {OUTPUT_DIR}…")
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

    print("\n✅  Training complete!")

main()

Loading dataset…


Generating train split: 0 examples [00:00, ? examples/s]

  Train: 540  |  Val: 60
Loading tokenizer…
Tokenising dataset…


Map:   0%|          | 0/540 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

...

trainable params: 589,824 || all params: 61,096,448 || trainable%: 0.9654

🚀  Starting training…


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,No log,8.848197
2,36.673218,8.595010
3,36.296875,7.897262
4,34.320074,6.879229
5,30.277505,6.049883
6,26.360608,5.163809
7,22.883299,4.351417
8,19.738634,3.603475
9,16.915744,3.025450
10,13.792155,2.529805


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector


💾  Saving LoRA adapter to /kaggle/working/lora-adapter…

✅  Training complete!


In [29]:
"""
inference.py – Kaggle Optimized Version
---------------------------------------
"""

import json
import torch
from transformers import T5ForConditionalGeneration, T5TokenizerFast
from peft import PeftModel

# ── Configuration (Strictly for Kaggle) ──────────────────────────────────────
BASE_MODEL_NAME = "t5-small"
ADAPTER_PATH    = "/kaggle/working/lora-adapter" 
MAX_INPUT_LEN   = 256
MAX_NEW_TOKENS  = 256  # Increased to ensure JSON isn't cut off
PREFIX          = "translate English to Calendar API: "
NUM_BEAMS       = 4
EARLY_STOPPING  = True

# ── Mock Calendar API ─────────────────────────────────────────────────────────

def execute_calendar_api(parsed_json: dict) -> None:
    action = parsed_json.get("action", "unknown").lower()
    print("\n" + "=" * 60)
    print("  📅  CALENDAR API CALL EXECUTED")
    print("=" * 60)

    if action == "insert":
        attendees = parsed_json.get("attendees", [])
        print(f"  ✅  ACTION   : Create Event")
        print(f"  📌  Title    : {parsed_json.get('title', 'N/A')}")
        print(f"  📅  Day      : {parsed_json.get('day', 'N/A')}")
        print(f"  ⏰  Time     : {parsed_json.get('start_time', 'N/A')}")
        print(f"  ⏱  Duration : {parsed_json.get('duration_minutes', 'N/A')} min")
        if attendees: print(f"  👥  Attendees: {', '.join(attendees)}")
    elif action == "delete":
        print(f"  🗑️   ACTION   : Delete Event\n  📌  Title    : {parsed_json.get('title', 'N/A')}")
    elif action == "reminder":
        print(f"  🔔  ACTION   : Set Reminder\n  📝  Task     : {parsed_json.get('task', 'N/A')}")
    else:
        print(f"  📦  Raw payload: {json.dumps(parsed_json, indent=4)}")
    
    print("=" * 60 + "\n")

# ── Model Loader (Kaggle Pathing) ─────────────────────────────────────────────

def load_model_and_tokenizer():
    print(f"Loading from {ADAPTER_PATH}...")
    tokenizer = T5TokenizerFast.from_pretrained(ADAPTER_PATH)
    base = T5ForConditionalGeneration.from_pretrained(BASE_MODEL_NAME)
    model = PeftModel.from_pretrained(base, ADAPTER_PATH)
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    return model, tokenizer, device

# ── Inference Logic ───────────────────────────────────────────────────────────
def run_inference(nl_command: str):
    model, tokenizer, device = load_model_and_tokenizer()
    
    input_text = PREFIX + nl_command
    # Explicitly add max_length=MAX_INPUT_LEN here
    inputs = tokenizer(
        input_text, 
        return_tensors="pt", 
        max_length=MAX_INPUT_LEN, 
        truncation=True, 
        padding=True
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs, 
            max_new_tokens=MAX_NEW_TOKENS, 
            num_beams=NUM_BEAMS, 
            early_stopping=EARLY_STOPPING
        )

    raw_output = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    print(f"Raw Output: {raw_output}")

    # ── UPDATED SALVAGE LOGIC ──
    if not raw_output.startswith("{") and '"action"' in raw_output:
        raw_output = "{" + raw_output
    if not raw_output.endswith("}") and '"action"' in raw_output:
        raw_output = raw_output + "}"

    parsed = None
    try:
        parsed = json.loads(raw_output)
    except json.JSONDecodeError:
        start = raw_output.find("{")
        end = raw_output.rfind("}") + 1
        if start != -1 and end > start:
            try:
                parsed = json.loads(raw_output[start:end])
            except:
                parsed = None

    if parsed:
        execute_calendar_api(parsed)
    else:
        print("❌ Could not generate valid JSON. Try training for more epochs.")
        
# ── TEST COMMAND ─────────────────────────────────────────────────────────────
run_inference("Schedule a meeting with Alice next Wednesday at 3pm")

Loading from /kaggle/working/lora-adapter...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Raw Output: "action":"insert","title":"Second","attendees":["Aleks"],"start_time":"15:00","duration_minutes":120,"day":"next Wednesday"

  📅  CALENDAR API CALL EXECUTED
  ✅  ACTION   : Create Event
  📌  Title    : Second
  📅  Day      : next Wednesday
  ⏰  Time     : 15:00
  ⏱  Duration : 120 min
  👥  Attendees: Aleks



In [30]:
run_inference("Cancel my standup tomorrow at 9am")

Loading from /kaggle/working/lora-adapter...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Raw Output: "action":"delete","title":"Standup","start_time":"9:00","day":"tomorrow"

  📅  CALENDAR API CALL EXECUTED
  🗑️   ACTION   : Delete Event
  📌  Title    : Standup



In [31]:
run_inference("Remind me to email Bob at 6pm today")

Loading from /kaggle/working/lora-adapter...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Raw Output: "action":"reminder","task":"email Bob","time":"18:00","day":"today"

  📅  CALENDAR API CALL EXECUTED
  🔔  ACTION   : Set Reminder
  📝  Task     : email Bob



In [32]:
run_inference("Move my review from 2pm to 4pm on next Friday")

Loading from /kaggle/working/lora-adapter...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Raw Output: "action":"update","title":"Review","old_start_time":"14:00","day":"next Friday"

  📅  CALENDAR API CALL EXECUTED
  📦  Raw payload: {
    "action": "update",
    "title": "Review",
    "old_start_time": "14:00",
    "day": "next Friday"
}



In [33]:
run_inference("Add Karen to my kickoff next Monday at 10am")

Loading from /kaggle/working/lora-adapter...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Raw Output: "action":"add_attendee","title":"Kara","start_time":"10:00","day":"next Monday"

  📅  CALENDAR API CALL EXECUTED
  📦  Raw payload: {
    "action": "add_attendee",
    "title": "Kara",
    "start_time": "10:00",
    "day": "next Monday"
}



In [34]:
run_inference("What meetings do I have tomorrow?")

Loading from /kaggle/working/lora-adapter...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Raw Output: "action":"query","day":"tomorrow"

  📅  CALENDAR API CALL EXECUTED
  📦  Raw payload: {
    "action": "query",
    "day": "tomorrow"
}



In [35]:
run_inference("Schedule a sync with Alice, Bob, and Carol tomorrow at 10am")

Loading from /kaggle/working/lora-adapter...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Raw Output: "action":"insert","title":"Sync","attendees":["Alice","Bob"],"start_time":"10:00","duration_minutes":120,"day":"tomorrow"

  📅  CALENDAR API CALL EXECUTED
  ✅  ACTION   : Create Event
  📌  Title    : Sync
  📅  Day      : tomorrow
  ⏰  Time     : 10:00
  ⏱  Duration : 120 min
  👥  Attendees: Alice, Bob



In [36]:
run_inference("Schedule a Dentist Appointment for next Tuesday at 10am")

Loading from /kaggle/working/lora-adapter...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Raw Output: "action":"insert","title":"dentist","attendees":["Dentist"],"start_time":"10:00","duration_minutes":60,"day":"next Tuesday"

  📅  CALENDAR API CALL EXECUTED
  ✅  ACTION   : Create Event
  📌  Title    : dentist
  📅  Day      : next Tuesday
  ⏰  Time     : 10:00
  ⏱  Duration : 60 min
  👥  Attendees: Dentist

